<a href="https://colab.research.google.com/github/ameliamazzola/miniproject2/blob/main/notebooks/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%cd /content
!rm -rf miniproject2
!git clone https://github.com/ameliamazzola/miniproject2.git

/content
Cloning into 'miniproject2'...
remote: Enumerating objects: 19980, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 19980 (delta 0), reused 0 (delta 0), pack-reused 19977 (from 1)
Receiving objects: 100% (19980/19980), 796.44 MiB | 26.80 MiB/s, done.
Resolving deltas: 100% (86/86), done.
Updating files: 100% (21663/21663), done.


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:00


In [4]:
import os
from ultralytics import YOLO

YAML_PATH = "miniproject2/yolo_dataset/data.yaml"
MODEL_DIR = "miniproject2/model"

os.makedirs(MODEL_DIR, exist_ok=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
!ls /content
!ls /content/miniproject2
!ls /content/miniproject2/yolo_dataset

miniproject2  sample_data
data.yaml  requirements.txt	 yolo_dataset
model	   src			 yolo_dataset_old_20260430_133514
notebooks  test_dataset.py	 yolo_dataset.yaml
outputs    test.ipynb
README.md  visualize_dataset.py
data.yaml  images  labels


In [6]:
import os

print("YAML_PATH:", os.path.abspath(YAML_PATH))
print("MODEL_DIR:", os.path.abspath(MODEL_DIR))

print("\nDoes YAML exist?", os.path.exists(YAML_PATH))
print("Does MODEL_DIR exist?", os.path.exists(MODEL_DIR))

YAML_PATH: /content/miniproject2/yolo_dataset/data.yaml
MODEL_DIR: /content/miniproject2/model

Does YAML exist? True
Does MODEL_DIR exist? True


In [7]:
print("Using dataset config:", YAML_PATH)

Using dataset config: miniproject2/yolo_dataset/data.yaml


In [8]:
!cat /content/miniproject2/yolo_dataset/data.yaml

path: /content/miniproject2/yolo_dataset
train: images/train
val: images/val

names:
  0: car
  1: bus
  2: van
  3: others


In [9]:
from pathlib import Path

img_dir = Path("/content/miniproject2/yolo_dataset/images/train")
lbl_dir = Path("/content/miniproject2/yolo_dataset/labels/train")

imgs = {p.stem for p in img_dir.glob("*.jpg")}
lbls = {p.stem for p in lbl_dir.glob("*.txt")}

print("Images:", len(imgs))
print("Labels:", len(lbls))
print("Matching:", len(imgs & lbls))

print("Example image:", list(imgs)[:5])
print("Example label:", list(lbls)[:5])

Images: 4636
Labels: 4636
Matching: 4636
Example image: ['MVI_39031_img01289', 'MVI_39031_img01186', 'MVI_39271_img00439', 'MVI_39271_img00158', 'MVI_20061_img00507']
Example label: ['MVI_39031_img01289', 'MVI_39031_img01186', 'MVI_39271_img00158', 'MVI_39271_img00439', 'MVI_20061_img00507']


In [10]:
!rm -f /content/miniproject2/yolo_dataset/labels/train.cache
!rm -f /content/miniproject2/yolo_dataset/labels/val.cache

In [11]:
from ultralytics import YOLO
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

DATA_YAML = "/content/miniproject2/yolo_dataset/data.yaml"
PROJECT_DIR = "/content/miniproject2/model"
RUN_NAME = "improved_model"

model = YOLO("yolov8s.pt")  # better than yolov8n, still reasonable on Colab T4

results = model.train(
    data=DATA_YAML,
    project=PROJECT_DIR,
    name=RUN_NAME,
    epochs=40,
    imgsz=768,
    batch=8,
    patience=15,
    device=0,

    # better learning behavior
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,

    # augmentations for traffic images
    mosaic=0.7,
    close_mosaic=10,
    mixup=0.05,
    scale=0.4,
    translate=0.08,
    fliplr=0.5,
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,

    # training output
    save=True,
    save_period=5,
    plots=True,
    exist_ok=True
)

# Best model path
best_model_path = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
print("Best model saved at:", best_model_path)

# Save backup to Google Drive so Colab reset does not delete it
drive_backup_dir = "/content/drive/MyDrive/miniproject2/model/improved_model"
os.makedirs(drive_backup_dir, exist_ok=True)

shutil.copy(best_model_path, f"{drive_backup_dir}/best.pt")
shutil.copy(f"{PROJECT_DIR}/{RUN_NAME}/weights/last.pt", f"{drive_backup_dir}/last.pt")

print("Backed up to Drive:", drive_backup_dir)

Mounted at /content/drive
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/miniproject2/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=improved_model, nbs=64, nms=False, opset=None, optimize=False, opt

In [12]:
from ultralytics import YOLO
from google.colab import files
import shutil, os

BEST_MODEL = "/content/drive/MyDrive/miniproject2/model/improved_model/best.pt"
DATA_YAML = "/content/miniproject2/yolo_dataset/data.yaml"

model = YOLO(BEST_MODEL)

metrics = model.val(
    data=DATA_YAML,
    split="val",
    plots=True,
    save_json=True,
    project="/content/miniproject2/model",
    name="final_eval",
    exist_ok=True
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

# Backup evaluation folder
shutil.copytree(
    "/content/miniproject2/model/final_eval",
    "/content/drive/MyDrive/miniproject2/model/final_eval",
    dirs_exist_ok=True
)

print("Evaluation saved to Drive.")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2554.3±1079.4 MB/s, size: 76.5 KB)
val: Scanning /content/miniproject2/yolo_dataset/labels/val.cache... 1431 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1431/1431 500.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 3.8it/s 23.6s
                   all       1431      14988      0.755      0.659      0.713      0.474
                   car       1350      11683      0.675      0.895      0.837      0.583
                   bus       1401       2299      0.806      0.462      0.612      0.354
                   van        775       1006      0.784       0.62       0.69      0.485
Speed: 1.4ms preprocess, 7.5ms inference, 0.0ms loss, 1.4ms postprocess per image
Saving /content/miniproject2/mod